# Aula 5 · Backpropagation na mão

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlexChequer/notebooks-insperai/blob/main/trainees/aula-05-backpropagation.ipynb)

**Trilha de Trainees — InsperAI** · o par prático da Aula 5

---

O vídeo do 3Blue1Brown mostrou a ideia do backpropagation. Aqui você faz a conta com números, linha por linha:

1. **Um neurônio só:** previsão, perda, derivadas e um passo.
2. **A rede do Igor:** ida (forward), volta (backward) e passo.
3. **O treino:** repetir ida, volta e passo com os 8 clientes da loja.
4. **Conferência:** comparar as suas derivadas com as do PyTorch.

**Como funciona:** as células com `# TODO` têm `...` no lugar do código. Troque cada `...` pelo que a dica pede e rode a célula. Logo abaixo sempre tem uma conferência que mostra ✅ ou ❌ com o número esperado. Se travar, a resposta está recolhida logo depois de cada exercício.

Rode as células na ordem, de cima para baixo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)


def confere(nome, obtido, esperado, tol=0.01):
    # Compara o que você calculou com o valor esperado.
    if obtido is None or obtido is ...:
        print(f"⬜ {nome}: ainda não preenchido")
        return False
    try:
        ok = np.allclose(np.asarray(obtido, dtype=float), np.asarray(esperado, dtype=float), atol=tol)
    except (TypeError, ValueError):
        ok = False
    if ok:
        print(f"✅ {nome}: {np.round(np.asarray(obtido, dtype=float), 3)}")
    else:
        print(f"❌ {nome}: deu {obtido}, esperado {esperado}")
    return ok

---
## Parte 1 · Um neurônio só

O caso mais simples possível: um neurônio com um peso `w` e um viés `b`, sem função de ativação.

$$\hat{y} = w \cdot x + b \qquad\qquad L = \tfrac{1}{2}(\hat{y} - y)^2$$

- `x = 2` é a entrada
- `w = 3` e `b = 0` são os números que a rede pode ajustar
- `y = 7` é a resposta certa

> **Ligando com o vídeo.** O 3Blue1Brown escreve o custo como $C = (a - y)^2$ e usa a sigmoide como ativação. Aqui o custo é $L = \tfrac{1}{2}(\hat{y} - y)^2$ (o ½ só serve para a derivada ficar limpa) e este primeiro neurônio não tem ativação. A regra da cadeia é exatamente a mesma.

In [ ]:
x = 2.0
w = 3.0
b = 0.0
y = 7.0

### 1.1 Ida: previsão e perda

In [ ]:
# TODO 1: a previsão do neurônio
y_chapeu = ...  # peso vezes entrada, mais o viés

# TODO 2: a perda (metade do erro ao quadrado)
perda = ...  # 0.5 * (previsão - resposta) ** 2

confere("previsão", y_chapeu, 6)
confere("perda", perda, 0.5)

<details><summary>Resposta</summary>

```python
# TODO 1: a previsão do neurônio
y_chapeu = w * x + b
# TODO 2: a perda (metade do erro ao quadrado)
perda = 0.5 * (y_chapeu - y) ** 2
```

</details>

### 1.2 Volta: as derivadas parciais

Queremos saber quanto a perda muda se mexermos **só** no `w`. O `w` não toca a perda direto: ele muda a previsão, e a previsão muda a perda. Pela **regra da cadeia**, multiplicamos os dois efeitos:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial w}$$

| derivada | quanto vale | em palavras |
|---|---|---|
| $\partial L / \partial \hat{y}$ | $\hat{y} - y$ | previsão − resposta |
| $\partial \hat{y} / \partial w$ | $x$ | o valor que passou pelo fio |
| $\partial \hat{y} / \partial b$ | $1$ | sempre 1 |

In [ ]:
# TODO 3: quanto a perda muda com a previsão
dL_dy = ...  # previsão - resposta

# TODO 4: quanto a perda muda com o peso (regra da cadeia)
dL_dw = ...  # dL_dy vezes o valor que passou pelo fio

# TODO 5: quanto a perda muda com o viés
dL_db = ...  # dL_dy vezes 1

confere("dL/dy", dL_dy, -1)
confere("dL/dw", dL_dw, -2)
confere("dL/db", dL_db, -1)

<details><summary>Resposta</summary>

```python
# TODO 3: quanto a perda muda com a previsão
dL_dy = y_chapeu - y
# TODO 4: quanto a perda muda com o peso (regra da cadeia)
dL_dw = dL_dy * x
# TODO 5: quanto a perda muda com o viés
dL_db = dL_dy * 1
```

</details>

**Conferindo com o cutucão.** A derivada diz: se `w` subir um pouquinho, a perda muda −2 vezes esse pouquinho. A célula abaixo testa isso mexendo 0,01 no `w`. Só rode.

In [ ]:
if dL_dw is not ...:
    w_cutucado = w + 0.01
    perda_cutucada = 0.5 * (w_cutucado * x + b - y) ** 2
    print(f"perda antes:  {0.5 * (w * x + b - y) ** 2:.4f}")
    print(f"perda depois: {perda_cutucada:.4f}")
    print(f"inclinação medida: {(perda_cutucada - 0.5) / 0.01:.2f}   |   sua derivada: {dL_dw}")

### 1.3 O passo

Cada número anda um pouco **contra** a sua derivada. O tamanho do passo é a taxa de aprendizado.

$$\text{novo} = \text{atual} - \text{taxa} \times \text{derivada}$$

In [ ]:
taxa = 0.1

# TODO 6: o novo peso e o novo viés
w_novo = ...  # w - taxa * dL_dw
b_novo = ...  # b - taxa * dL_db

# Ida de novo, com os números novos (já pronto)
if w_novo is not ... and b_novo is not ...:
    y_novo = w_novo * x + b_novo
    perda_nova = 0.5 * (y_novo - y) ** 2
else:
    y_novo = perda_nova = ...

confere("w novo", w_novo, 3.2)
confere("b novo", b_novo, 0.1)
confere("previsão nova", y_novo, 6.5)
confere("perda nova", perda_nova, 0.125)

<details><summary>Resposta</summary>

```python
# TODO 6: o novo peso e o novo viés
w_novo = w - taxa * dL_dw
b_novo = b - taxa * dL_db
```

</details>

Um passo e a perda caiu de **0,5** para **0,125**. Isso é o gradiente descendente da A1, agora com a derivada calculada pela regra da cadeia.

---
## Parte 2 · A rede do Igor

Agora uma rede pequena de verdade, no exemplo de tamanho de roupa da última aula: uma loja quer sugerir P, M ou G a partir do peso do cliente. O cliente da vez é o Igor, 70 kg, que veste M.

A rede tem 1 entrada, 2 neurônios no meio (A e B) com ReLU e 3 saídas (P, M, G) com softmax.

```
            ┌── A ──┬── P
  peso ── x ┤       ├── M
            └── B ──┴── G
```

- A entrada é o peso numa escala pequena: `x = (peso − 50) / 10`. Para o Igor (70 kg), `x = 2`.
- A resposta certa vira três números: `[0, 1, 0]` quer dizer "é M".
- Os pesos começam **chutados**. Treinar é ajustá-los.

In [ ]:
# Pesos chutados
W1 = np.array([-1.0, 2.0])        # fios x→A e x→B
b1 = np.array([3.0, -3.0])        # vieses de A e B
W2 = np.array([[1.0, 1.0],        # fios A→P e B→P
               [1.0, -1.0],       # fios A→M e B→M
               [-1.0, 1.0]])      # fios A→G e B→G
b2 = np.array([0.0, 0.0, 0.0])    # vieses de P, M, G

x_igor = (70 - 50) / 10           # 2.0
y_igor = np.array([0, 1, 0])      # o Igor veste M

### 2.1 Ida (forward)

Três contas, na ordem:

1. `z1`: cada neurônio do meio faz **entrada × peso + viés**
2. `a1`: a **ReLU** zera o que for negativo e deixa passar o que for positivo
3. `z2`: cada saída soma A e B, cada um vezes o peso do seu fio, mais o viés

O softmax (que transforma os scores em probabilidades) já vem pronto.

**Dicas de NumPy**
- `W1 * x` multiplica cada número de `W1` por `x`.
- `np.maximum(z, 0)` troca os negativos por 0.
- `W2 @ a1` faz, para cada saída, a soma "A × peso + B × peso".

In [ ]:
def relu(z):
    # TODO 7: negativos viram 0, positivos passam
    return ...  # use np.maximum(z, 0)


def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()


def forward(x, W1, b1, W2, b2):
    # TODO 8: a conta dos neurônios do meio
    z1 = ...  # entrada vezes peso, mais viés
    # TODO 9: a ReLU
    a1 = ...  # aplique a relu em z1
    # TODO 10: a conta das saídas
    z2 = ...  # W2 @ a1, mais o viés b2
    p = softmax(z2) if z2 is not ... else ...
    return z1, a1, z2, p


z1, a1, z2, p = forward(x_igor, W1, b1, W2, b2)
confere("A e B", a1, [1, 1])
confere("scores P, M, G", z2, [2, 0, 0])
confere("probabilidades", p, [0.787, 0.106, 0.106])

<details><summary>Resposta</summary>

```python
def relu(z):
    # TODO 7: negativos viram 0, positivos passam
    return np.maximum(z, 0)
def softmax(z):
def forward(x, W1, b1, W2, b2):
    # TODO 8: a conta dos neurônios do meio
    z1 = W1 * x + b1
    # TODO 9: a ReLU
    a1 = relu(z1)
    # TODO 10: a conta das saídas
    z2 = W2 @ a1 + b2
```

</details>

A rede aposta em **P com 79%**, mas o Igor veste M.

A **perda** mede esse erro com um número só. Para classificação ela é `−ln(probabilidade dada à resposta certa)`: quanto menos probabilidade para a resposta certa, maior a perda.

In [ ]:
def perda_de(p, y):
    return -np.log(p[y.argmax()])


if p is not ...:
    print(f"perda do Igor: {perda_de(p, y_igor):.2f}   (esperado: 2,24)")

### 2.2 Volta (backward)

Aqui entram duas regras. Cada neurônio faz `z = peso × valor + viés`, e dessa conta saem duas derivadas:

| regra | derivada | conta |
|---|---|---|
| **do fio** (ajusta o peso) | $\partial z / \partial \text{peso} = \text{valor}$ | culpa de onde o fio chega × valor que passou por ele |
| **do neurônio** (manda a culpa para trás) | $\partial z / \partial \text{valor} = \text{peso}$ | soma de (culpa da frente × peso do fio) |

Chamamos de **culpa** a derivada parcial da perda em relação a um número da rede (no vídeo, é o quanto aquele número "quer" ser empurrado). A ordem é de trás para a frente:

1. `d2`: culpa das saídas = previsão − resposta
2. `dW2`: culpa dos fios que chegam nas saídas (regra do fio)
3. `da1`: culpa de A e B (regra do neurônio)
4. `d1`: a culpa passa pela ReLU (neurônio desligado zera)
5. `dW1`: culpa dos fios da entrada (regra do fio de novo)

**Dicas de NumPy**
- `np.outer(d2, a1)` monta a tabela "cada culpa de saída × cada valor de A e B".
- `W2.T @ d2` faz, para A e para B, a soma "culpa × peso" pelos três fios.
- `(z1 > 0)` vale 1 onde o neurônio está ligado e 0 onde está desligado.

In [ ]:
def backward(x, y, z1, a1, p, W2):
    # TODO 11: culpa das saídas
    d2 = ...  # previsão - resposta
    # Regra do fio (exemplo pronto): culpa da chegada × valor que passou
    dW2 = np.outer(d2, a1) if d2 is not ... else ...
    db2 = d2
    # TODO 12: regra do neurônio, a culpa de A e B
    da1 = ...  # W2.T @ d2
    # TODO 13: a culpa passa pela ReLU
    d1 = ...  # da1 vezes (z1 > 0)
    # TODO 14: regra do fio para os fios da entrada
    dW1 = ...  # culpa do neurônio vezes o valor que passou (x)
    db1 = d1
    return dW1, db1, dW2, db2, da1


dW1, db1, dW2, db2, da1 = backward(x_igor, y_igor, z1, a1, p, W2)
confere("culpa das saídas", db2, [0.787, -0.894, 0.106])
confere("culpa de A e B", da1, [-0.213, 1.787])
confere("culpa dos fios x→A e x→B", dW1, [-0.426, 3.574])
confere("culpa dos fios das saídas", dW2, [[0.787, 0.787], [-0.894, -0.894], [0.106, 0.106]])

<details><summary>Resposta</summary>

```python
def backward(x, y, z1, a1, p, W2):
    # TODO 11: culpa das saídas
    d2 = p - y
    # TODO 12: regra do neurônio, a culpa de A e B
    da1 = W2.T @ d2
    # TODO 13: a culpa passa pela ReLU
    d1 = da1 * (z1 > 0)
    # TODO 14: regra do fio para os fios da entrada
    dW1 = d1 * x
```

</details>

Repare: B leva **+1,79** e A leva **−0,21**, mesmo valendo os dois 1, porque os pesos dos fios que saem deles são diferentes.

### 2.3 O passo

Todos os pesos e vieses andam contra a sua culpa, ao mesmo tempo.

In [ ]:
def passo(W1, b1, W2, b2, dW1, db1, dW2, db2, taxa=0.1):
    W1 = W1 - taxa * dW1  # exemplo pronto
    # TODO 15: faça o mesmo para os outros três
    b1 = ...  # b1 - taxa * db1
    W2 = ...  # W2 - taxa * dW2
    b2 = ...  # b2 - taxa * db2
    return W1, b1, W2, b2


if dW1 is ... or db1 is ... or db2 is ...:
    print("⬜ Preencha a volta (2.2) antes de dar o passo.")
else:
    W1n, b1n, W2n, b2n = passo(W1, b1, W2, b2, dW1, db1, dW2, db2)
    confere("novos fios x→A e x→B", W1n, [-0.957, 1.643])
    confere("novos vieses de A e B", b1n, [3.021, -3.179])

    # Mesmo Igor, de novo pela rede, com os números novos
    if b2n is not ... and W2n is not ... and b1n is not ...:
        _, a1n, _, pn = forward(x_igor, W1n, b1n, W2n, b2n)
        confere("A e B depois do passo", a1n, [1.106, 0.106])
        confere("probabilidades depois do passo", pn, [0.435, 0.510, 0.055])
        print(f"perda: {perda_de(p, y_igor):.2f}  →  {perda_de(pn, y_igor):.2f}")

<details><summary>Resposta</summary>

```python
def passo(W1, b1, W2, b2, dW1, db1, dW2, db2, taxa=0.1):
    # TODO 15: faça o mesmo para os outros três
    b1 = b1 - taxa * db1
    W2 = W2 - taxa * dW2
    b2 = b2 - taxa * db2
```

</details>

Depois de **um passo** a rede já aposta em M (51%).

---
## Parte 3 · O treino

Um passo com uma pessoa não é treino. Treinar é repetir **ida → volta → passo** com todos os clientes, muitas vezes.

A classe abaixo só junta as funções que você escreveu. Não tem lacuna: leia e rode.

In [ ]:
class RedeNeural:
    def __init__(self):
        self.W1 = np.array([-1.0, 2.0])
        self.b1 = np.array([3.0, -3.0])
        self.W2 = np.array([[1.0, 1.0], [1.0, -1.0], [-1.0, 1.0]])
        self.b2 = np.zeros(3)

    def prever(self, x):
        return forward(x, self.W1, self.b1, self.W2, self.b2)[3]

    def treinar_um_passo(self, xs, ys, taxa=0.1):
        soma = [0, 0, 0, 0]
        perda = 0
        for x, y in zip(xs, ys):
            z1, a1, z2, p = forward(x, self.W1, self.b1, self.W2, self.b2)   # ida
            perda += perda_de(p, y)
            dW1, db1, dW2, db2, _ = backward(x, y, z1, a1, p, self.W2)       # volta
            soma = [s + d for s, d in zip(soma, (dW1, db1, dW2, db2))]
        media = [s / len(xs) for s in soma]                                   # culpa média dos clientes
        self.W1, self.b1, self.W2, self.b2 = passo(self.W1, self.b1, self.W2, self.b2, *media, taxa=taxa)
        return perda / len(xs)

In [ ]:
# Os 8 clientes da loja
nomes = ["Ana", "Bruno", "Carol", "Davi", "Igor", "Elisa", "Fábio", "Gabi"]
pesos = np.array([50, 55, 62, 68, 70, 75, 84, 90])
tamanhos = np.array([0, 0, 1, 1, 1, 1, 2, 2])      # 0 = P, 1 = M, 2 = G

xs = (pesos - 50) / 10
ys = np.eye(3)[tamanhos]                           # vira [1,0,0], [0,1,0] ou [0,0,1]

rede = RedeNeural()
historico = []
treinou = False
try:
    for i in range(300):
        historico.append(rede.treinar_um_passo(xs, ys, taxa=0.1))
    treinou = True
except Exception as erro:
    print("⬜ O treino não rodou. Confira se todos os TODOs da Parte 2 estão preenchidos.")
    print("   Erro:", erro)

if treinou:
    print(f"perda média: {historico[0]:.2f}  →  {historico[-1]:.2f}")
    print()
    acertos = 0
    for nome, kg, x, t in zip(nomes, pesos, xs, tamanhos):
        p_cliente = rede.prever(x)
        aposta = "PMG"[p_cliente.argmax()]
        certo = "PMG"[t]
        acertos += aposta == certo
        print(f"{nome:6s} {kg} kg   aposta {aposta} ({p_cliente.max():.0%})   veste {certo}   {'✅' if aposta == certo else '❌'}")
    print(f"\nacertos: {acertos}/8")

In [ ]:
if not treinou:
    print("⬜ Rode o treino da célula anterior primeiro.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(historico, color="#8B5CF6", lw=2.5)
    ax[0].set_title("Perda média ao longo do treino")
    ax[0].set_xlabel("passo")
    ax[0].set_ylabel("perda")

    kg = np.linspace(45, 95, 200)
    probs = np.array([rede.prever((k - 50) / 10) for k in kg])
    for i, (nome, cor) in enumerate(zip("PMG", ["#8B5CF6", "#10B981", "#F59E0B"])):
        ax[1].plot(kg, probs[:, i], color=cor, lw=2.5, label=nome)
    ax[1].scatter(pesos, np.full(8, -0.06), c=[["#8B5CF6", "#10B981", "#F59E0B"][t] for t in tamanhos], s=70, zorder=3)
    ax[1].set_title("Probabilidade de cada tamanho, por peso (bolinhas: clientes)")
    ax[1].set_xlabel("peso (kg)")
    ax[1].legend()
    plt.tight_layout()
    plt.show()

Ninguém escolheu esses pesos na mão. Os cortes entre P, M e G apareceram sozinhos, só repetindo ida, volta e passo.

---
## Parte 4 · Conferindo com o PyTorch

No PyTorch, toda a volta que você escreveu é uma linha: `perda.backward()`. A célula abaixo refaz a rede do Igor no PyTorch e compara as derivadas dele com as suas. Só rode.

In [ ]:
try:
    import torch

    tW1 = torch.tensor([-1.0, 2.0], requires_grad=True)
    tb1 = torch.tensor([3.0, -3.0], requires_grad=True)
    tW2 = torch.tensor([[1.0, 1.0], [1.0, -1.0], [-1.0, 1.0]], requires_grad=True)
    tb2 = torch.zeros(3, requires_grad=True)

    ta1 = torch.relu(tW1 * 2.0 + tb1)
    tz2 = tW2 @ ta1 + tb2
    perda_t = -torch.log_softmax(tz2, dim=0)[1]     # o Igor veste M (posição 1)
    perda_t.backward()                              # a volta inteira

    confere("fios da entrada   (seu × PyTorch)", dW1, tW1.grad.numpy())
    confere("vieses de A e B   (seu × PyTorch)", db1, tb1.grad.numpy())
    confere("fios das saídas   (seu × PyTorch)", dW2, tW2.grad.numpy())
    confere("vieses das saídas (seu × PyTorch)", db2, tb2.grad.numpy())
except ImportError:
    print("PyTorch não está instalado aqui. No Google Colab ele já vem pronto.")

---
## Desafios

1. **Taxa de aprendizado.** Na Parte 3, troque `taxa=0.1` por `0.01` e depois por `2.0`. O que acontece com a curva da perda em cada caso?
2. **Outro cliente.** Refaça a ida e a volta da Parte 2 para a Gabi (90 kg, veste G: `y = np.array([0, 0, 1])`). Quem leva mais culpa agora, A ou B?
3. **Neurônio desligado.** Faça a ida para a Ana (50 kg, `x = 0`). O neurônio B fica desligado. Olhe `dW1` e `db1` depois da volta: qual é a culpa do fio x→B e do viés de B? Por quê?
4. **Pesos aleatórios.** Na classe `RedeNeural`, troque os pesos chutados por `np.random.randn(2)` e `np.random.randn(3, 2)`. A rede sempre chega a 8/8? Rode algumas vezes.